# Quantum Phase Estimation

Given a unitary $U$ and an eigenstate $|u\rangle$ with
$U|u\rangle = e^{2\pi i \phi}|u\rangle$, QPE estimates $\phi$
to $n$ bits of precision using:

1. Prepare $n$ precision qubits in $|+\rangle^{\otimes n}$.
2. Apply controlled-$U^{2^k}$ for $k = 0, \dots, n-1$.
3. Apply the inverse QFT to read $\phi$ as a binary fraction.

We estimate the eigenphase of the **T gate** ($U = T$, $\phi = 1/8$)
using 2–4 precision qubits.

In [ ]:
import math
import qiskit as qk
import qiskit_aer as qka

## QFT and inverse QFT

In [ ]:
def qft_circuit(n):
    qc = qk.QuantumCircuit(n, name="QFT")
    for j in range(n):
        qc.h(j)
        for k in range(j):
            qc.cp(math.pi / 2 ** (j - k), k, j)
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc

def inverse_qft_circuit(n):
    qc = qk.QuantumCircuit(n, name="IQFT")
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    for j in range(n - 1, -1, -1):
        for k in range(j - 1, -1, -1):
            qc.cp(-math.pi / 2 ** (j - k), k, j)
        qc.h(j)
    return qc

print(inverse_qft_circuit(3).draw())

## QPE circuit (T gate, phase = 1/8)

Layout: q0..q_{n-1} = precision qubits, q_n = eigenstate $|1\rangle$.

In [ ]:
n_prec = 3
n = n_prec + 1

qc = qk.QuantumCircuit(n)
qc.x(n_prec)  # eigenstate |1>
qc.h(range(n_prec))  # precision in |+>^n

# controlled powers of T
for j in range(n_prec):
    for _ in range(2**j):
        qc.cp(math.pi / 4, j, n_prec)

qc.compose(inverse_qft_circuit(n_prec), inplace=True)
print(qc.draw())

## Statevector result (no shot noise)

In [ ]:
sv = qk.quantum_info.Statevector.from_instruction(qc)
for state, p in sorted(sv.probabilities_dict().items()):
    k = int(state[::-1], 2)
    phase_est = k / (2**n_prec)
    print(f"|{state}>  k={k:2d}  phase={phase_est:.4f}  (= {phase_est*2*math.pi:.4f} rad)  p={p:.4f}")

## Scaling: more precision qubits

In [ ]:
for n_prec in (2, 3, 4):
    n = n_prec + 1
    qc2 = qk.QuantumCircuit(n)
    qc2.x(n_prec)
    qc2.h(range(n_prec))
    for j in range(n_prec):
        for _ in range(2**j):
            qc2.cp(math.pi / 4, j, n_prec)
    qc2.compose(inverse_qft_circuit(n_prec), inplace=True)
    sv2 = qk.quantum_info.Statevector.from_instruction(qc2)
    top = max(sv2.probabilities_dict().items(), key=lambda kv: kv[1])
    k = int(top[0][::-1], 2)
    print(f"n={n_prec}: top state |{top[0]}>  k={k}  phase={k/(2**n_prec):.4f}  p={top[1]:.4f}")

## Sampling QPE with 3 precision qubits

In [ ]:
qc_meas = qk.QuantumCircuit(n_prec + 1, n_prec)
qc_meas.x(n_prec)
qc_meas.h(range(n_prec))
for j in range(n_prec):
    for _ in range(2**j):
        qc_meas.cp(math.pi / 4, j, n_prec)
qc_meas.compose(inverse_qft_circuit(n_prec), inplace=True)
qc_meas.measure(range(n_prec), range(n_prec))
print(qc_meas.draw())

sim = qka.AerSimulator()
counts = sim.run(qk.transpile(qc_meas, sim), shots=4096).result().get_counts()
print("\nshot histogram (top 5):")
for bits, cnt in sorted(counts.items(), key=lambda kv: -kv[1])[:5]:
    k = int(bits, 2)
    print(f"  |{bits}>  k={k}  phase={k/(2**n_prec):.4f}  ({cnt:4d} shots)")

The dominant outcome is $|001\rangle = 1$, giving phase estimate
$1/8 = 0.125$, which is exactly the T-gate eigenphase.